# SNIES 2020–2025 — Normalización, EDA y dimensiones de programas

**Objetivo.** Construir una base de trabajo reproducible para el proyecto de análisis de mercado sin incorporar todavía CECOS. El flujo parte del histórico SNIES, normaliza los cambios de estructura entre años, realiza un EDA, extrae los programas de la Ibero (código IES **2830**), genera una primera homologación automática de nombres de programas, prepara una base para homologación manual y construye la dimensión de universidades.

### Salidas esperadas
1. `01_Ibero_2830_programas.xlsx`
2. `02_Dim_Programa_Agrupacion_Automatica.xlsx`
3. `03_Programas_Normalizados_Manual.xlsx`
4. `04_Dim_Universidades.xlsx`
5. `05_EDA_SNIES.xlsx`

> La homologación automática es una **primera propuesta**. Los matches de confianza media/baja deben revisarse manualmente antes de usarlos como regla productiva.

## 1. Configuración del proyecto

La estructura esperada es:

```text
03_Data_Preparation
├─ notebook/
│  └─ 01_SNIES_Normalizacion_EDA_Dimensiones.ipynb
└─ data/
   ├─ raw/
   │  └─ Historico_SNIES.xlsx
   └─ processed/
```

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import csv
import gzip
import html
import json
import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

# Salidas y visualización
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt


# CONFIGURACIÓN DEL PROYECTO

CWD = Path.cwd().resolve()

# Busca hacia arriba la raíz que contiene data/raw
PROJECT = next(
    (
        carpeta
        for carpeta in [CWD, *CWD.parents]
        if (carpeta / "data" / "raw").exists()
    ),
    None
)

if PROJECT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta raíz del proyecto con data/raw."
    )

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

INPUT = RAW / "Historico_SNIES.xlsx"

CODIGO_IBERO = "2830"

# Solo activar si queremos exportar el histórico normalizado completo.
EXPORTAR_HISTORICO_NORMALIZADO = False


# ============================================================
# VALIDACIÓN
# ============================================================

assert INPUT.exists(), f"No se encontró: {INPUT.resolve()}"

print("Raíz del proyecto:", PROJECT)
print("Archivo fuente:", INPUT)
print("Carpeta de salidas:", PROCESSED)


Raíz del proyecto: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics
Archivo fuente: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\raw\Historico_SNIES.xlsx
Carpeta de salidas: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed


## 2. Carga y homologación estructural del histórico SNIES

Se cargan automáticamente todas las hojas cuyo nombre corresponde a un año.

El esquema de 2025 se utiliza como referencia canónica para homologar
las columnas históricas. Las nuevas columnas incorporadas en años
posteriores se conservan y se agregan con valor 0 a los años en los
que estructuralmente no existían.

AÑO_ORIGEN se conserva como campo de trazabilidad.

In [2]:
# 2.1 DETECCIÓN AUTOMÁTICA DE AÑOS

excel_snies = pd.ExcelFile(INPUT)

ANIOS = sorted(
    hoja
    for hoja in excel_snies.sheet_names
    if re.fullmatch(r"\d{4}", str(hoja))
)

print("Años detectados:", ANIOS)

Años detectados: ['2020', '2021', '2022', '2023', '2024', '2025']


In [3]:
# 2.2 ESQUEMA CANÓNICO 2025

COLUMNAS_2025 = list(
    pd.read_excel(
        INPUT,
        sheet_name="2025",
        nrows=0
    ).columns
)

# Columna residual eliminada también en Power Query
COLUMNAS_2025 = [
    columna
    for columna in COLUMNAS_2025
    if columna != "Column42"
]

print("Columnas esperadas según 2025:", len(COLUMNAS_2025))

Columnas esperadas según 2025: 41


In [4]:
# 2.3 COLUMNAS RENOMBRADAS MANUALES

RENOMBRES_COLUMNAS = {
    "IES_PADRE": "IES PADRE",
    "PRINCIPAL O SECCIONAL": "TIPO IES",
    "ID CARACTER": "ID CARÁCTER IES",
    "CARACTER IES": "CARÁCTER IES",
    "CÓDIGO DEL MUNICIPIO (IES)": "CÓDIGO DEL MUNICIPIO IES",
    "ID METODOLOGÍA": "ID MODALIDAD",
    "METODOLOGÍA": "MODALIDAD",
    "ID ÁREA DE CONOCIMIENTO": "ID ÁREA",
    "ID CINE CODIGO DETALLADO": "ID CINE CAMPO DETALLADO",
    "DESC CINE CODIGO DETALLADO": "DESC CINE CAMPO DETALLADO",
    "PRIMER CURSO": "MATRICULADOS PRIMER CURSO",
}

## Se lee únicamente los encabezados de una hoja y aplica los mismos renombres definidos en Power Query

def obtener_columnas_homologadas(anio):

    columnas = list(
        pd.read_excel(
            INPUT,
            sheet_name=anio,
            nrows=0
        ).columns
    )

    # Equivalente a MissingField.Ignore
    columnas = [
        RENOMBRES_COLUMNAS.get(columna, columna)
        for columna in columnas
    ]

    # Column42 es residual y en Power Query se elimina
    columnas = [
        columna
        for columna in columnas
        if columna != "Column42"
    ]

    return columnas

In [5]:
# 2.4 DIAGNOSTICO

validacion_columnas = []

for anio in ANIOS:

    columnas_anio = obtener_columnas_homologadas(anio)

    faltantes = [
        columna
        for columna in COLUMNAS_2025
        if columna not in columnas_anio
    ]

    adicionales = [
        columna
        for columna in columnas_anio
        if columna not in COLUMNAS_2025
    ]

    validacion_columnas.append({
        "AÑO": anio,
        "COLUMNAS_AÑO": len(columnas_anio),
        "COLUMNAS_2025": len(COLUMNAS_2025),
        "FALTANTES": len(faltantes),
        "ADICIONALES": len(adicionales),
        "DETALLE_FALTANTES": faltantes,
        "DETALLE_ADICIONALES": adicionales,
    })

df_validacion_columnas = pd.DataFrame(validacion_columnas)

df_validacion_columnas

,AÑO,COLUMNAS_AÑO,COLUMNAS_2025,FALTANTES,ADICIONALES,DETALLE_FALTANTES,DETALLE_ADICIONALES
0,2020,39,41,2,0,"[IES ACREDITADA, PROGRAMA ACREDITADO]",[]
1,2021,41,41,0,0,[],[]
2,2022,41,41,0,0,[],[]
3,2023,41,41,0,0,[],[]
4,2024,41,41,0,0,[],[]
5,2025,41,41,0,0,[],[]


In [6]:
# 2.5 VALIDACIÓN DE PARES 

def validar_renombres(anio):
    """
    Muestra qué columnas originales fueron renombradas
    y cuál es su nombre final.
    """

    columnas_originales = list(
        pd.read_excel(
            INPUT,
            sheet_name=anio,
            nrows=0
        ).columns
    )

    registros = []

    for columna in columnas_originales:

        columna_final = RENOMBRES_COLUMNAS.get(
            columna,
            columna
        )

        registros.append({
            "AÑO": anio,
            "COLUMNA_ORIGINAL": columna,
            "COLUMNA_FINAL": columna_final,
            "RENOMBRADA": columna != columna_final,
            "EXISTE_EN_2025": columna_final in COLUMNAS_2025,
        })

    return pd.DataFrame(registros)

df_mapeo_columnas = pd.concat(
    [
        validar_renombres(anio)
        for anio in ANIOS
    ],
    ignore_index=True
)

df_mapeo_columnas[
    df_mapeo_columnas["RENOMBRADA"]
]

,AÑO,COLUMNA_ORIGINAL,COLUMNA_FINAL,RENOMBRADA,EXISTE_EN_2025
1,2020,IES_PADRE,IES PADRE,True,True
3,2020,PRINCIPAL O SECCIONAL,TIPO IES,True,True
6,2020,ID CARACTER,ID CARÁCTER IES,True,True
7,2020,CARACTER IES,CARÁCTER IES,True,True
10,2020,CÓDIGO DEL MUNICIPIO (IES),CÓDIGO DEL MUNICIPIO IES,True,True
18,2020,ID METODOLOGÍA,ID MODALIDAD,True,True
19,2020,METODOLOGÍA,MODALIDAD,True,True
20,2020,ID ÁREA DE CONOCIMIENTO,ID ÁREA,True,True
28,2020,ID CINE CODIGO DETALLADO,ID CINE CAMPO DETALLADO,True,True
29,2020,DESC CINE CODIGO DETALLADO,DESC CINE CAMPO DETALLADO,True,True


In [7]:
# 2.6 CONSOLIDADO AÑO

def cargar_anio(anio):
    """
    Carga una hoja SNIES y replica las transformaciones
    estructurales realizadas en Power Query.
    """

    df = pd.read_excel(
        INPUT,
        sheet_name=anio
    )

    # 1. Homologar nombres de columnas
    df = df.rename(
        columns=RENOMBRES_COLUMNAS
    )

    # 2. Eliminar columna residual si existe
    df = df.drop(
        columns=["Column42"],
        errors="ignore"
    )

    # 3. Caso específico de 2020:
    #    estas columnas no existían en la fuente original
    if anio == "2020":

        if "IES ACREDITADA" not in df.columns:
            df["IES ACREDITADA"] = pd.NA

        if "PROGRAMA ACREDITADO" not in df.columns:
            df["PROGRAMA ACREDITADO"] = pd.NA

    # 4. Campo de trazabilidad
    df["AÑO_ORIGEN"] = int(anio)

    return df

historicos = {
    anio: cargar_anio(anio)
    for anio in ANIOS
}

In [8]:
# 2.7 VALIDACIÓN DE ESTRUCTURA POR AÑO

validacion_estructura = []

for anio, df in historicos.items():

    columnas_anio = [
        columna
        for columna in df.columns
        if columna != "AÑO_ORIGEN"
    ]

    faltantes = [
        columna
        for columna in COLUMNAS_2025
        if columna not in columnas_anio
    ]

    adicionales = [
        columna
        for columna in columnas_anio
        if columna not in COLUMNAS_2025
    ]

    validacion_estructura.append({
        "AÑO": anio,
        "COLUMNAS_AÑO": len(columnas_anio),
        "COLUMNAS_2025": len(COLUMNAS_2025),
        "FALTANTES": len(faltantes),
        "ADICIONALES": len(adicionales),
        "DETALLE_FALTANTES": faltantes,
        "DETALLE_ADICIONALES": adicionales,
    })

df_validacion_estructura = pd.DataFrame(
    validacion_estructura
)

df_validacion_estructura

,AÑO,COLUMNAS_AÑO,COLUMNAS_2025,FALTANTES,ADICIONALES,DETALLE_FALTANTES,DETALLE_ADICIONALES
0,2020,41,41,0,0,[],[]
1,2021,41,41,0,0,[],[]
2,2022,41,41,0,0,[],[]
3,2023,41,41,0,0,[],[]
4,2024,41,41,0,0,[],[]
5,2025,41,41,0,0,[],[]


In [9]:

# 2.8 CONSOLIDACIÓN DEL HISTÓRICO

df_historico = pd.concat(
    [
        historicos[anio]
        for anio in ANIOS
    ],
    ignore_index=True
)

print(
    f"Histórico consolidado: "
    f"{len(df_historico):,} filas x "
    f"{len(df_historico.columns)} columnas"
)

Histórico consolidado: 308,579 filas x 42 columnas


In [10]:
df_historico.tail()

,CÓDIGO DE LA INSTITUCIÓN,IES PADRE,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),TIPO IES,ID SECTOR IES,SECTOR IES,ID CARÁCTER IES,CARÁCTER IES,CÓDIGO DEL DEPARTAMENTO (IES),DEPARTAMENTO DE DOMICILIO DE LA IES,...,CÓDIGO DEL MUNICIPIO (PROGRAMA),MUNICIPIO DE OFERTA DEL PROGRAMA,ID SEXO,SEXO,AÑO,SEMESTRE,MATRICULADOS PRIMER CURSO,IES ACREDITADA,PROGRAMA ACREDITADO,AÑO_ORIGEN
308574,La información suministrada corresponde a lo r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308575,"A partir de 2021, la Universidad Internacional...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308576,"Para el año 2025, de las 305 IES con personerí...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308577,A partir de la información de la vigencia 2019...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308578,La información de acreditación de IES y progra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [11]:
# 2.9 REGISTROS VÁLIDOS PARA LOS COMENTARIOS QUE DESCARGA EL REPORTE
filas_antes = len(df_historico)

codigo_ies_numerico = pd.to_numeric(
    df_historico["CÓDIGO DE LA INSTITUCIÓN"],
    errors="coerce"
)

df_historico = df_historico[
    codigo_ies_numerico.notna()
].copy()

filas_despues = len(df_historico)

print(f"Registros antes del filtro: {filas_antes:,}")
print(f"Registros válidos:          {filas_despues:,}")
print(f"Registros eliminados:       {filas_antes - filas_despues:,}")

Registros antes del filtro: 308,579
Registros válidos:          308,555
Registros eliminados:       24


In [12]:
df_historico.tail()

,CÓDIGO DE LA INSTITUCIÓN,IES PADRE,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),TIPO IES,ID SECTOR IES,SECTOR IES,ID CARÁCTER IES,CARÁCTER IES,CÓDIGO DEL DEPARTAMENTO (IES),DEPARTAMENTO DE DOMICILIO DE LA IES,...,CÓDIGO DEL MUNICIPIO (PROGRAMA),MUNICIPIO DE OFERTA DEL PROGRAMA,ID SEXO,SEXO,AÑO,SEMESTRE,MATRICULADOS PRIMER CURSO,IES ACREDITADA,PROGRAMA ACREDITADO,AÑO_ORIGEN
308566,9941,9941.0,CORPORACIÓN INSTITUCIÓN UNIVERSITARIA SALUD CO...,Principal,2.0,Privado,3.0,Institución Universitaria/Escuela Tecnológica,73.0,Tolima,...,73001.0,Ibagué,2.0,Femenino,2025.0,2.0,20.0,No,No,2025
308567,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,1.0,Masculino,2025.0,2.0,11.0,Si,No,2025
308568,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,2.0,Femenino,2025.0,2.0,9.0,Si,No,2025
308569,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,1.0,Masculino,2025.0,2.0,7.0,Si,No,2025
308570,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,2.0,Femenino,2025.0,2.0,3.0,Si,No,2025


## 3. Construcción de la dimensión de universidades

A partir del histórico SNIES consolidado (`df_historico`), se construye una dimensión única de instituciones de educación superior para ser utilizada posteriormente en el modelo de Power BI.

La dimensión tendrá como llave principal el código de institución SNIES (`ID_UNIVERSIDAD`) y contendrá una única fila por universidad.

El proceso incluye:

- normalización del código institucional;
- limpieza del nombre de la institución;
- revisión de variaciones históricas del nombre;
- selección del nombre institucional más reciente disponible;
- validación de unicidad de la llave;
- exportación del archivo `04_Dim_Universidades.xlsx`.

El objetivo es obtener una dimensión estable y reutilizable que permita relacionar el histórico SNIES con los atributos institucionales en el modelo analítico.

In [13]:
# 3.1 FUNCIONES DE NORMALIZACIÓN

def normalizar_codigo(serie):
    """
    Convierte códigos numéricos almacenados como texto o decimal
    a texto entero limpio.
    
    """

    numerico = pd.to_numeric(
        serie,
        errors="coerce"
    )

    return (
        numerico
        .astype("Int64")
        .astype("string")
    )


def normalizar_texto(serie):
    """
    Limpieza básica de campos de texto sin modificar
    su contenido semántico.

    - elimina BOM
    - normaliza Unicode
    - elimina espacios repetidos
    - elimina espacios iniciales/finales
    """

    return (
        serie
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.replace("<U+FEFF>", "", regex=False)
        .str.replace("\xa0", " ", regex=False)
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [14]:
# 3.2 NORMALIZACIÓN DEL CÓDIGO DE UNIVERSIDAD

df_historico["CÓDIGO DE LA INSTITUCIÓN"] = normalizar_codigo(
    df_historico["CÓDIGO DE LA INSTITUCIÓN"]
)

# validación 

print(
    "Universidades únicas:",
    df_historico["CÓDIGO DE LA INSTITUCIÓN"].nunique()
)

print(
    "Códigos nulos:",
    df_historico["CÓDIGO DE LA INSTITUCIÓN"].isna().sum()
)

print(
    "Ibero 2830 encontrada:",
    CODIGO_IBERO in set(
        df_historico["CÓDIGO DE LA INSTITUCIÓN"].dropna()
    )
)

Universidades únicas: 336
Códigos nulos: 0
Ibero 2830 encontrada: True


In [15]:
# 3.3 BASE PARA DIMENSIÓN DE UNIVERSIDADES

df_universidades_base = df_historico[
    [
        "CÓDIGO DE LA INSTITUCIÓN",
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        "AÑO_ORIGEN",
    ]
].copy()

# Normalización del nombre de la institución
df_universidades_base[
    "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
] = normalizar_texto(
    df_universidades_base[
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
    ]
)

#VALIDACIÓN
variaciones_nombre = (
    df_universidades_base
    .groupby(
        "CÓDIGO DE LA INSTITUCIÓN"
    )["INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Instituciones con más de un nombre:",
    (variaciones_nombre > 1).sum()
)

variaciones_nombre.head(20)

Instituciones con más de un nombre: 49


CÓDIGO DE LA INSTITUCIÓN
3828    3
9903    3
2743    3
9131    2
1206    2
4817    2
4811    2
4837    2
9102    2
4111    2
4110    2
4107    2
4106    2
3821    2
3808    2
4112    2
4835    2
3703    2
3725    2
1215    2
Name: INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES), dtype: int64

In [16]:
# 3.4  DETALLE DE INSTITUCIONES CON VARIACIONES

codigos_con_variaciones = variaciones_nombre[
    variaciones_nombre > 1
].index

df_variaciones_universidad = (
    df_universidades_base[
        df_universidades_base[
            "CÓDIGO DE LA INSTITUCIÓN"
        ].isin(codigos_con_variaciones)
    ]
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    )
    .reset_index(drop=True)
)

df_variaciones_universidad.head(10)

,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)
0,1206,UNIVERSIDAD DE NARI¿O
1,1206,UNIVERSIDAD DE NARIÑO
2,1214,UNIVERSIDAD DE CUNDINAMARCA
3,1214,UNIVERSIDAD DE CUNDINAMARCA-UDEC
4,1215,UNIVERSIDAD DE CUNDINAMARCA
5,1215,UNIVERSIDAD DE CUNDINAMARCA-UDEC
6,1216,UNIVERSIDAD DE CUNDINAMARCA
7,1216,UNIVERSIDAD DE CUNDINAMARCA-UDEC
8,1833,CORPORACIÓN UNIVERSIDAD DEL SINU - ELIAS BECHA...
9,1833,UNIVERSIDAD DEL SINU - ELIAS BECHARA ZAINUM - ...


In [17]:
# 3.5 VALIDACIÓN DE CARACTERES SOSPECHOSOS

df_nombres_sospechosos = (
    df_universidades_base[
        df_universidades_base[
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
        ].str.contains(
            r"[¿�]",
            regex=True,
            na=False
        )
    ]
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
            "AÑO_ORIGEN",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "AÑO_ORIGEN",
        ]
    )
    .reset_index(drop=True)
)

df_nombres_sospechosos

,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),AÑO_ORIGEN
0,1206,UNIVERSIDAD DE NARI¿O,2020
1,2707,FUNDACI¿N UNIVERSITARIA JUAN N. CORPAS,2020
2,2719,UNIVERSIDAD CAT¿LICA LUIS AMIG¿,2020
3,2731,FUNDACION UNIVERSITARIA CATOLICA LUMEN GENTIUM...,2020
4,2745,FUNDACI¿N UNIVERSITARIA COMPENSAR,2020
5,3828,CORPORACI¿N TECNOL¿GICA DE EDUCACI¿N SUPERIOR ...,2020
6,4835,CORPORACION UNIVERSITARIA TALLER CINCO CENTRO ...,2020
7,9131,FUNDACI¿N UNIVERSITARIA CERVANTES SAN AGUST¿N ...,2020
8,9914,"ESEIT ¿ ESCUELA SUPERIOR DE EMPRESA, INGENIER¿...",2020
9,9929,Universidad Aut¿noma Ind¿gena Intercultural - ...,2020


In [18]:
# 3.6 SELECCIÓN DE LA DENOMINACIÓN DE LA IES

# Identificar nombres con posibles problemas de codificación
df_universidades_base["NOMBRE_SOSPECHOSO"] = (
    df_universidades_base[
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
    ]
    .str.contains(
        r"[¿�ÃÂ]",
        regex=True,
        na=False
    )
)
# Estandarizar denominaciones de IES en mayúscula
df_universidades_base[
    "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
] = (
    df_universidades_base[
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
    ]
    .str.upper()
)

# CANDIDATOS DE NOMBRE POR INSTITUCIÓN Y AÑO

df_candidatos_ies = (
    df_universidades_base
    .groupby(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
            "AÑO_ORIGEN",
            "NOMBRE_SOSPECHOSO",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="FRECUENCIA")
)


# CANDIDATOS VÁLIDOS

df_candidatos_validos = df_candidatos_ies[
    ~df_candidatos_ies["NOMBRE_SOSPECHOSO"]
].copy()

df_candidatos_validos = (
    df_candidatos_validos
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "AÑO_ORIGEN",
            "FRECUENCIA",
        ],
        ascending=[
            True,
            False,
            False,
        ]
    )
)

dim_universidades = (
    df_candidatos_validos
    .drop_duplicates(
        subset="CÓDIGO DE LA INSTITUCIÓN",
        keep="first"
    )
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    ]
    .sort_values(
        "CÓDIGO DE LA INSTITUCIÓN"
    )
    .reset_index(drop=True)
)

dim_universidades.head(20) 



,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)
0,1101,UNIVERSIDAD NACIONAL DE COLOMBIA
1,1102,UNIVERSIDAD NACIONAL DE COLOMBIA
2,1103,UNIVERSIDAD NACIONAL DE COLOMBIA
3,1104,UNIVERSIDAD NACIONAL DE COLOMBIA
4,1105,UNIVERSIDAD PEDAGOGICA NACIONAL
5,1106,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
6,1107,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
7,1108,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
8,1109,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
9,1110,UNIVERSIDAD DEL CAUCA


In [19]:
col_codigo = 'CÓDIGO DE LA INSTITUCIÓN'
col_ies = 'INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)'

print("Total de registros:", len(dim_universidades))

print("Códigos únicos:",
      dim_universidades[col_codigo].nunique())

print("Instituciones únicas:",
      dim_universidades[col_ies].nunique())

print("Instituciones con varios códigos:",
      (dim_universidades.groupby(col_ies)[col_codigo]
       .nunique() > 1).sum())



Total de registros: 336
Códigos únicos: 336
Instituciones únicas: 289
Instituciones con varios códigos: 21


In [20]:
# 3.7 EXPORTACIÓN DIM_UNIVERSIDADES

OUTPUT_DIM_UNIVERSIDADES = (
    PROCESSED / "01_Dim_Universidades.xlsx"
)

dim_universidades.to_excel(
    OUTPUT_DIM_UNIVERSIDADES,
    sheet_name="Dim_Universidades",
    index=False
)

print("Archivo generado correctamente:")
print(OUTPUT_DIM_UNIVERSIDADES)

Archivo generado correctamente:
C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed\01_Dim_Universidades.xlsx


## 4. Construcción de la dimensión de programas SNIES

A partir del histórico SNIES consolidado (`df_historico`) se construye un catálogo único de programas académicos.

La dimensión utiliza `CÓDIGO SNIES DEL PROGRAMA` como llave y conserva una única denominación canónica de `PROGRAMA ACADÉMICO` por código.

El proceso contempla:

- normalización del código SNIES;
- limpieza básica de las denominaciones académicas;
- identificación de variaciones históricas de nombre;
- detección de problemas de codificación;
- corrección manual de códigos sin ninguna denominación válida;
- selección de la denominación válida más reciente;
- validación de unicidad y cobertura.

La normalización realizada en esta sección no homologa todavía programas equivalentes entre diferentes códigos SNIES. Esa homologación se realizará posteriormente tomando como referencia los programas de la Ibero.

In [21]:
# 4.1 PREPARACIÓN DE LA BASE DE PROGRAMAS

COL_CODIGO_PROGRAMA = "CÓDIGO SNIES DEL PROGRAMA"
COL_PROGRAMA = "PROGRAMA ACADÉMICO"

PATRON_CODIFICACION_DANADA = r"[¿�ÃÂ]"


# Normalizar la llave directamente en el histórico
df_historico[COL_CODIGO_PROGRAMA] = normalizar_codigo(
    df_historico[COL_CODIGO_PROGRAMA]
)


# Construir base de trabajo
df_programas_base = df_historico[
    [
        COL_CODIGO_PROGRAMA,
        COL_PROGRAMA,
        "AÑO_ORIGEN",
    ]
].copy()


# Conservar registros con código y nombre
df_programas_base = df_programas_base[
    df_programas_base[COL_CODIGO_PROGRAMA].notna()
    & df_programas_base[COL_PROGRAMA].notna()
].copy()


# Limpieza básica del nombre
df_programas_base[COL_PROGRAMA] = normalizar_texto(
    df_programas_base[COL_PROGRAMA]
)


# Eliminar textos vacíos
df_programas_base = df_programas_base[
    df_programas_base[COL_PROGRAMA].ne("")
].copy()


print(
    "Códigos SNIES únicos:",
    df_programas_base[COL_CODIGO_PROGRAMA].nunique()
)

Códigos SNIES únicos: 15564


In [22]:
# 4.2 CLAVE NORMALIZADA Y DIAGNÓSTICO DE CALIDAD

def crear_clave_programa(serie):
    """
    Genera una clave auxiliar para comparar denominaciones.

    La clave:
    - elimina diferencias de tildes;
    - convierte a mayúsculas;
    - normaliza puntuación;
    - compacta espacios.

    No reemplaza el nombre visible del programa.
    """

    return (
        serie
        .astype("string")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.upper()
        .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


df_programas_base["PROGRAMA_KEY_ORIGINAL"] = crear_clave_programa(
    df_programas_base[COL_PROGRAMA]
)


df_programas_base["NOMBRE_SOSPECHOSO_ORIGINAL"] = (
    df_programas_base[COL_PROGRAMA]
    .str.contains(
        PATRON_CODIFICACION_DANADA,
        regex=True,
        na=False
    )
)

### Nota sobre variaciones y nombres con codificación dañada

En esta etapa se revisan las variaciones históricas de nombre asociadas a cada `CÓDIGO SNIES DEL PROGRAMA`.

- Se identificaron **15.564 códigos SNIES únicos**.
- **301 códigos** presentan más de una denominación observada.
- **105 códigos** tienen al menos un nombre con problemas de codificación, por ejemplo caracteres como `¿` o `�`.
- La mayoría de esos códigos también tiene una denominación válida en otro año.
- Solo **14 códigos** no tienen ninguna denominación válida disponible en el histórico.

Por esta razón, únicamente esos 14 códigos requieren una corrección manual explícita.

La tabla de diagnóstico muestra:

- `NOMBRES_OBSERVADOS`: cantidad de nombres distintos encontrados para un mismo código.
- `NOMBRES_NORMALIZADOS`: cantidad de variantes que quedan después de eliminar diferencias superficiales como tildes o puntuación.
- `NOMBRES_SOSPECHOSOS`: cantidad de denominaciones distintas con problemas de codificación.
- `REGISTROS_SOSPECHOSOS`: cantidad de filas históricas donde aparece alguna denominación dañada.

Los nombres dañados no se corrigen mediante reemplazos generales, ya que el carácter `¿` puede representar letras diferentes. Se prioriza una versión válida disponible en otro año y solo se corrigen manualmente los códigos sin ninguna alternativa válida.

In [23]:
# DIAGNÓSTICO DE VARIACIONES

df_programas_base["PROGRAMA_SOSPECHOSO"] = (
    df_programas_base[COL_PROGRAMA]
    .where(
        df_programas_base[
            "NOMBRE_SOSPECHOSO_ORIGINAL"
        ]
    )
)


diagnostico_programas = (
    df_programas_base
    .groupby(COL_CODIGO_PROGRAMA)
    .agg(
        NOMBRES_OBSERVADOS=(
            COL_PROGRAMA,
            "nunique"
        ), #nombres literalmente diferentes encontrados
        NOMBRES_NORMALIZADOS=(
            "PROGRAMA_KEY_ORIGINAL",
            "nunique"
        ), #variantes después de quitar tildes/puntuación/etc.
        NOMBRES_SOSPECHOSOS=(
            "PROGRAMA_SOSPECHOSO",
            "nunique"
        ), #nombres dañados distintos
        REGISTROS_SOSPECHOSOS=(
            "NOMBRE_SOSPECHOSO_ORIGINAL",
            "sum"
        ), #cantidad de filas históricas afectadas
    )
    .sort_values(
        [
            "NOMBRES_NORMALIZADOS",
            "NOMBRES_OBSERVADOS",
        ],
        ascending=False
    )
)


print(
    "Códigos con más de una denominación observada:",
    (
        diagnostico_programas[
            "NOMBRES_OBSERVADOS"
        ] > 1
    ).sum()
)

print(
    "Códigos afectados por codificación:",
    (
        diagnostico_programas[
            "NOMBRES_SOSPECHOSOS"
        ] > 0
    ).sum()
)

diagnostico_programas.head(30)

Códigos con más de una denominación observada: 301
Códigos afectados por codificación: 105


,NOMBRES_OBSERVADOS,NOMBRES_NORMALIZADOS,NOMBRES_SOSPECHOSOS,REGISTROS_SOSPECHOSOS
CÓDIGO SNIES DEL PROGRAMA,,,,
10110,3,3,1,12
20886,3,3,1,8
19866,4,2,1,2
106143,3,2,0,0
10711,3,2,1,2
11452,3,2,1,27
12503,3,2,1,6
12505,3,2,1,8
19109,3,2,1,8


In [24]:
# 4.3 COBERTURA DE DENOMINACIONES VÁLIDAS

codigos_totales = set(
    df_programas_base[
        COL_CODIGO_PROGRAMA
    ].dropna()
)


codigos_con_nombre_valido = set(
    df_programas_base.loc[
        ~df_programas_base[
            "NOMBRE_SOSPECHOSO_ORIGINAL"
        ],
        COL_CODIGO_PROGRAMA
    ].dropna()
)


codigos_sin_nombre_valido = sorted(
    codigos_totales
    - codigos_con_nombre_valido
)


print(
    "Códigos SNIES totales:",
    len(codigos_totales)
)

print(
    "Códigos sin ninguna denominación válida:",
    len(codigos_sin_nombre_valido)
)

codigos_sin_nombre_valido

Códigos SNIES totales: 15564
Códigos sin ninguna denominación válida: 14


['101414',
 '102058',
 '102319',
 '102478',
 '102554',
 '103648',
 '103707',
 '105161',
 '111728',
 '14295',
 '15955',
 '2134',
 '5274',
 '91157']

In [25]:
# DETALLE PARA REVISIÓN MANUAL

columnas_revision = [
    COL_CODIGO_PROGRAMA,
    COL_PROGRAMA,
    "AÑO_ORIGEN",
    "CÓDIGO DE LA INSTITUCIÓN",
    "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
    "NIVEL DE FORMACIÓN",
    "MODALIDAD",
]


df_revision_manual_programas = (
    df_historico[
        df_historico[
            COL_CODIGO_PROGRAMA
        ].isin(codigos_sin_nombre_valido)
    ][columnas_revision]
    .drop_duplicates()
    .sort_values(
        [
            COL_CODIGO_PROGRAMA,
            "AÑO_ORIGEN",
        ]
    )
    .reset_index(drop=True)
)


df_revision_manual_programas

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,AÑO_ORIGEN,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),NIVEL DE FORMACIÓN,MODALIDAD
0,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2020,2745,FUNDACI¿N UNIVERSITARIA COMPENSAR,Tecnológica,Distancia (virtual)
1,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2021,2745,FUNDACIÓN UNIVERSITARIA COMPENSAR,Tecnológico,Virtual
2,102058,TECNOLOGIA EN GESTI¿N DE EXPORTACIONES E IMPOR...,2020,2836,CORPORACION UNIVERSITARIA EMPRESARIAL DE SALAM...,Tecnológica,Presencial
3,102319,MAESTR¿A EN CALIDAD EN SERVICIOS DE SALUD,2020,1804,UNIVERSIDAD AUTONOMA DEL CARIBE- UNIAUTONOMA,Maestría,Distancia (virtual)
4,102478,MAESTRIA EN SALUD P¿BLICA,2020,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Maestría,Presencial
5,102478,MAESTRIA EN SALUD P¿BLICA,2021,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Maestría,Presencial
6,102554,LICENCIATURA EN PEDAGOG¿A INFANTIL,2020,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Universitaria,Distancia (virtual)
7,103648,ADMINISTRACI¿N DE LA SALUD OCUPACIONAL,2020,9905,FUNDACION ESCUELA TECNOLOGICA DE NEIVA - JESUS...,Universitaria,Presencial
8,103648,ADMINISTRACI¿N DE LA SALUD OCUPACIONAL,2021,9905,FUNDACION ESCUELA TECNOLOGICA DE NEIVA - JESUS...,Universitario,Presencial
9,103707,TECNOLOGIA EN GESTI¿N DE SISTEMAS MECATRONICOS,2020,3706,FUNDACION CENTRO COLOMBIANO DE ESTUDIOS PROFES...,Tecnológica,Presencial


In [26]:
# 4.4 CORRECCIONES MANUALES DE CODIFICACIÓN

CORRECCIONES_MANUALES_PROGRAMAS = {
    "101414": "TECNOLOGÍA EN GESTIÓN DE PROCESOS ADMINISTRATIVOS",
    "102058": "TECNOLOGÍA EN GESTIÓN DE EXPORTACIONES E IMPORTACIONES",
    "102319": "MAESTRÍA EN CALIDAD EN SERVICIOS DE SALUD",
    "102478": "MAESTRÍA EN SALUD PÚBLICA",
    "102554": "LICENCIATURA EN PEDAGOGÍA INFANTIL",
    "103648": "ADMINISTRACIÓN DE LA SALUD OCUPACIONAL",
    "103707": "TECNOLOGÍA EN GESTIÓN DE SISTEMAS MECATRÓNICOS",
    "105161": "ESPECIALIZACIÓN EN INNOVACIÓN EDUCATIVA DIGITAL",
    "111728": "TÉCNICA PROFESIONAL EN PROCESOS GRÁFICOS",
    "14295": "ESPECIALIZACIÓN EN VIOLENCIA INTRAFAMILIAR: NIÑEZ Y ADOLESCENCIA",
    "15955": "TECNOLOGÍA EN SISTEMAS DE INFORMACIÓN",
    "2134": "TECNOLOGÍA EN PRODUCCIÓN INDUSTRIAL",
    "5274": "TECNOLOGÍA EN GESTIÓN CONTABLE Y FINANCIERA",
    "91157": "TECNOLOGÍA EN CONTABILIDAD Y FINANZAS",
}

#campo canónico sin perder el original
df_programas_base[
    "PROGRAMA_ACADÉMICO_CANÓNICO"
] = (
    df_programas_base[
        COL_CODIGO_PROGRAMA
    ]
    .map(CORRECCIONES_MANUALES_PROGRAMAS)
    .fillna(
        df_programas_base[
            COL_PROGRAMA
        ]
    )
)

#Controles
df_programas_base["NOMBRE_SOSPECHOSO"] = (
    df_programas_base[
        "PROGRAMA_ACADÉMICO_CANÓNICO"
    ]
    .str.contains(
        PATRON_CODIFICACION_DANADA,
        regex=True,
        na=False
    )
)


df_programas_base["PROGRAMA_KEY"] = crear_clave_programa(
    df_programas_base[
        "PROGRAMA_ACADÉMICO_CANÓNICO"
    ]
)

#Validación 
codigos_con_nombre_valido = set(
    df_programas_base.loc[
        ~df_programas_base[
            "NOMBRE_SOSPECHOSO"
        ],
        COL_CODIGO_PROGRAMA
    ].dropna()
)


codigos_sin_nombre_valido = sorted(
    codigos_totales
    - codigos_con_nombre_valido
)

print(
    "Registros históricos todavía sospechosos:",
    df_programas_base[
        "NOMBRE_SOSPECHOSO"
    ].sum()
)

print(
    "Códigos sin denominación válida:",
    len(codigos_sin_nombre_valido)
)


assert len(codigos_sin_nombre_valido) == 0, (
    "Existen códigos SNIES sin una denominación válida."
)

#Pueden haber mas registros porque el dato histórico dañado se conserva como evidencia, pero ya existe una versión válida para seleccionar en la dimensión.

Registros históricos todavía sospechosos: 751
Códigos sin denominación válida: 0


In [27]:
# 4.5 SELECCIÓN DE LA DENOMINACIÓN CANÓNICA

# Estandarizar las denominaciones finales en mayúscula
df_programas_base[
    "PROGRAMA_ACADÉMICO_CANÓNICO"
] = (
    df_programas_base[
        "PROGRAMA_ACADÉMICO_CANÓNICO"
    ]
    .str.upper()
)

df_candidatos_programa = (
    df_programas_base[
        ~df_programas_base[
            "NOMBRE_SOSPECHOSO"
        ]
    ]
    .groupby(
        [
            COL_CODIGO_PROGRAMA,
            "PROGRAMA_ACADÉMICO_CANÓNICO",
            "AÑO_ORIGEN",
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "FRECUENCIA"
        }
    )
)

#orden 
df_candidatos_programa = (
    df_candidatos_programa
    .sort_values(
        [
            COL_CODIGO_PROGRAMA,
            "AÑO_ORIGEN",
            "FRECUENCIA",
            "PROGRAMA_ACADÉMICO_CANÓNICO",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ]
    )
)

#eliminamos los duplicados finales

dim_programas = (
    df_candidatos_programa
    .drop_duplicates(
        subset=COL_CODIGO_PROGRAMA,
        keep="first"
    )
    [
        [
            COL_CODIGO_PROGRAMA,
            "PROGRAMA_ACADÉMICO_CANÓNICO",
        ]
    ]
    .rename(
        columns={
            "PROGRAMA_ACADÉMICO_CANÓNICO":
                COL_PROGRAMA
        }
    )
    .reset_index(drop=True)
)

dim_programas["_ORDEN_CODIGO"] = pd.to_numeric(
    dim_programas[
        COL_CODIGO_PROGRAMA
    ],
    errors="coerce"
)


dim_programas = (
    dim_programas
    .sort_values(
        [
            "_ORDEN_CODIGO",
            COL_CODIGO_PROGRAMA,
        ]
    )
    .drop(
        columns="_ORDEN_CODIGO"
    )
    .reset_index(drop=True)
)

dim_programas.tail(20)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO
15544,118231,ESPECIALIZACION EN EDUCACIÓN PARA LA INCLUSIÓN...
15545,118237,ESPECIALIZACIÓN EN DERECHO CONSTITUCIONAL
15546,118251,MAESTRIA EN DERECHO
15547,118257,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS
15548,118263,ESPECIALIZACIÓN EN INGENIERÍA FERROVIARIA
15549,118269,TÉCNICO PROFESIONAL EN ANÁLISIS Y REPORTE DE D...
15550,118272,TECNOLOGIA EN PROCESAMIENTO Y CONSERVACION DE ...
15551,118285,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...
15552,118286,MAESTRÍA EN ADMINISTRACIÓN (MBA)
15553,118289,ENFERMERÍA


In [28]:
# 4.6 VALIDACIÓN FINAL DE DIM_PROGRAMAS

total_codigos_origen = (
    df_programas_base[
        COL_CODIGO_PROGRAMA
    ]
    .nunique()
)

total_codigos_dimension = (
    dim_programas[
        COL_CODIGO_PROGRAMA
    ]
    .nunique()
)

nombres_sospechosos_dimension = (
    dim_programas[
        COL_PROGRAMA
    ]
    .str.contains(
        PATRON_CODIFICACION_DANADA,
        regex=True,
        na=False
    )
    .sum()
)


assert dim_programas[
    COL_CODIGO_PROGRAMA
].notna().all(), (
    "Existen códigos SNIES nulos."
)


assert dim_programas[
    COL_CODIGO_PROGRAMA
].is_unique, (
    "Existen códigos SNIES duplicados."
)


assert dim_programas[
    COL_PROGRAMA
].notna().all(), (
    "Existen programas sin denominación."
)


assert (
    total_codigos_origen
    == total_codigos_dimension
), (
    "No todos los códigos SNIES llegaron a la dimensión."
)


assert nombres_sospechosos_dimension == 0, (
    "La dimensión final contiene nombres con codificación sospechosa."
)


print("✓ Código SNIES sin nulos.")
print("✓ Código SNIES único.")
print("✓ Nombre de programa sin nulos.")
print("✓ Sin caracteres sospechosos en la dimensión.")
print(
    "✓ Códigos en dimensión:",
    total_codigos_dimension
)

✓ Código SNIES sin nulos.
✓ Código SNIES único.
✓ Nombre de programa sin nulos.
✓ Sin caracteres sospechosos en la dimensión.
✓ Códigos en dimensión: 15564


In [29]:
# 4.7 EXPORTACIÓN DIM_PROGRAMAS

OUTPUT_DIM_PROGRAMAS = (
    PROCESSED / "02_Dim_Programas_SNIES.xlsx"
)

dim_programas.to_excel(
    OUTPUT_DIM_PROGRAMAS,
    sheet_name="Dim_Programas",
    index=False
)

print("Archivo generado correctamente:")
print(OUTPUT_DIM_PROGRAMAS)

Archivo generado correctamente:
C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed\02_Dim_Programas_SNIES.xlsx


## 5. Homologación de programas contra el portafolio Ibero

Se utiliza la Iberoamericana (`CÓDIGO DE LA INSTITUCIÓN = 2830`) como catálogo de referencia para identificar programas equivalentes en otras instituciones. Posteriormente se compara este catálogo con la base institucional vigente a 2026.

Los programas históricos se conservan sin modificaciones. De la base 2026 únicamente se incorporan códigos que no existían previamente en el histórico.

Los programas que aún no cuentan con código oficial (`POR DEFINIR`) también se incorporan al catálogo, ya que forman parte del portafolio vigente de la Ibero. Para mantener una llave única mientras se asigna el código SNIES/SIET definitivo, se les genera un identificador temporal `PD_`.

La homologación se realiza en tres niveles:

1. **EXACTA:** el nombre normalizado del programa coincide exactamente con un programa Ibero.
2. **VALIDAR:** no existe coincidencia exacta, pero se identifica un programa Ibero con alta similitud como candidato para revisión manual.
3. **SIN_EQUIVALENCIA_IBERO:** después de la validación no se identifica un programa equivalente dentro del portafolio Ibero.

El proceso conserva siempre el código y nombre original del programa SNIES y agrega el código y nombre del programa Ibero homologado cuando corresponda.

In [36]:
# 5.1 CATÁLOGO HISTÓRICO DE PROGRAMAS IBERO

codigos_programas_ibero = (
    df_historico.loc[
        df_historico[
            "CÓDIGO DE LA INSTITUCIÓN"
        ] == CODIGO_IBERO,
        "CÓDIGO SNIES DEL PROGRAMA"
    ]
    .dropna()
    .unique()
)


programas_ibero_historico = (
    dim_programas[
        dim_programas[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].isin(codigos_programas_ibero)
    ]
    [
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
        ]
    ]
    .rename(
        columns={
            "CÓDIGO SNIES DEL PROGRAMA":
                "CÓDIGO PROGRAMA IBERO",

            "PROGRAMA ACADÉMICO":
                "PROGRAMA IBERO",
        }
    )
    .copy()
)


print(
    "Programas históricos Ibero:",
    len(programas_ibero_historico)
)

Programas históricos Ibero: 62


In [37]:
# 5.2 BASE INSTITUCIONAL IBERO 2026

INPUT_IBERO_2026 = RAW / "Programas_Ibero.xlsx"

df_ibero_2026 = pd.read_excel(
    INPUT_IBERO_2026,
    sheet_name="Hoja1"
)


# Conservar el código recibido de la fuente
df_ibero_2026["CÓDIGO_ORIGINAL_2026"] = (
    df_ibero_2026[
        "CÓDIGO SNIES / CÓDIGO SIET"
    ]
    .astype("string")
    .str.strip()
)


# Normalizar nombre del programa
df_ibero_2026["PROGRAMA IBERO"] = (
    normalizar_texto(
        df_ibero_2026[
            "NOMBRE DEL PROGRAMA"
        ]
    )
    .str.upper()
)


# Identificar programas pendientes de código oficial
df_ibero_2026["CODIGO_POR_DEFINIR"] = (
    df_ibero_2026["CÓDIGO_ORIGINAL_2026"]
    .str.upper()
    .eq("POR DEFINIR")
)


# Normalizar códigos oficiales
df_ibero_2026["CÓDIGO PROGRAMA IBERO"] = normalizar_codigo(
    df_ibero_2026[
        "CÓDIGO SNIES / CÓDIGO SIET"
    ]
)

In [38]:
# 5.3 LLAVES TEMPORALES PARA PROGRAMAS POR DEFINIR

mask_por_definir = (
    df_ibero_2026["CODIGO_POR_DEFINIR"]
)


# Generar una llave temporal reproducible a partir del nombre
claves_temporales = (
    crear_clave_programa(
        df_ibero_2026.loc[
            mask_por_definir,
            "PROGRAMA IBERO"
        ]
    )
    .str.replace(" ", "_", regex=False)
    .str[:60]
    .radd("PD_")
)


df_ibero_2026.loc[
    mask_por_definir,
    "CÓDIGO PROGRAMA IBERO"
] = claves_temporales


df_ibero_2026["ESTADO_CODIGO"] = (
    df_ibero_2026["CODIGO_POR_DEFINIR"]
    .map({
        True: "POR DEFINIR",
        False: "DEFINIDO",
    })
)

In [39]:
# 5.4 CATÁLOGO IBERO 202

programas_ibero_2026 = (
    df_ibero_2026[
        [
            "CÓDIGO PROGRAMA IBERO",
            "PROGRAMA IBERO",
            "ESTADO_CODIGO",
        ]
    ]
    .drop_duplicates(
        subset="CÓDIGO PROGRAMA IBERO"
    )
    .reset_index(drop=True)
)


print(
    "Programas 2026:",
    len(programas_ibero_2026)
)

print(
    "Con código definido:",
    (
        programas_ibero_2026[
            "ESTADO_CODIGO"
        ] == "DEFINIDO"
    ).sum()
)

print(
    "Por definir:",
    (
        programas_ibero_2026[
            "ESTADO_CODIGO"
        ] == "POR DEFINIR"
    ).sum()
)

programas_ibero_2026

Programas 2026: 69
Con código definido: 64
Por definir: 5


,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,ESTADO_CODIGO
0,110927,ADMINISTRACION EN SALUD,DEFINIDO
1,995,DERECHO,DEFINIDO
2,2056,ESPECIALIZACION EN AUDIOLOGIA,DEFINIDO
3,111072,ESPECIALIZACION EN AUDITORIA EN SALUD,DEFINIDO
4,2054,FISIOTERAPIA,DEFINIDO
...,...,...,...
64,PD_MAESTRIA_EN_GERENCIA_DE_PROYECTOS,MAESTRÍA EN GERENCIA DE PROYECTOS,POR DEFINIR
65,119272,DOCTORADO EN EDUCACIÓN,DEFINIDO
66,69836,TÉCNICO LABORAL EN AUXILIAR EN ENFERMERÍA,DEFINIDO
67,PD_MAESTRIA_EN_INGENIERIA_DE_SOFTWARE,MAESTRÍA EN INGENIERÍA DE SOFTWARE,POR DEFINIR


In [40]:
# 5.5 PROGRAMAS NUEVOS FRENTE AL HISTÓRICO

codigos_historicos_ibero = set(
    programas_ibero_historico[
        "CÓDIGO PROGRAMA IBERO"
    ]
)


programas_nuevos_2026 = (
    programas_ibero_2026[
        (
            programas_ibero_2026[
                "ESTADO_CODIGO"
            ] == "POR DEFINIR"
        )
        |
        (
            ~programas_ibero_2026[
                "CÓDIGO PROGRAMA IBERO"
            ].isin(codigos_historicos_ibero)
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Programas a incorporar desde 2026:",
    len(programas_nuevos_2026)
)

programas_nuevos_2026

Programas a incorporar desde 2026: 15


,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,ESTADO_CODIGO
0,995,DERECHO,DEFINIDO
1,111338,ESPECIALIZACION EN TELESALUD,DEFINIDO
2,110778,LICENCIATURA EN EDUCACION ARTISTICA,DEFINIDO
3,1710,MAESTRÍA EN ADMINISTRACIÓN DE EMPRESAS - MBA,DEFINIDO
4,118223,ESPECIALIZACIÓN EN DOCENCIA UNIVERSITARIA,DEFINIDO
5,1798,ESPECIALIZACIÓN EN REVISORÍA FISCAL Y AUDITORÍ...,DEFINIDO
6,118427,ESPECIALIZACIÓN EN GERENCIA DEL TALENTO HUMANO,DEFINIDO
7,119276,ESPECIALIZACIÓN EN DERECHO ADMINISTRATIVO,DEFINIDO
8,PD_MAESTRIA_EN_GERENCIA_DE_LA_SEGURIDAD_Y_SALU...,MAESTRÍA EN GERENCIA DE LA SEGURIDAD Y SALUD E...,POR DEFINIR
9,PD_DISENO_VISUAL,DISEÑO VISUAL,POR DEFINIR


In [41]:
# 5.6 CATÁLOGO IBERO ACTUALIZADO

programas_ibero_historico["ESTADO_CODIGO"] = "DEFINIDO"


programas_ibero = pd.concat(
    [
        programas_ibero_historico,
        programas_nuevos_2026,
    ],
    ignore_index=True
)


assert programas_ibero[
    "CÓDIGO PROGRAMA IBERO"
].notna().all(), (
    "Existen programas Ibero sin llave."
)


assert programas_ibero[
    "CÓDIGO PROGRAMA IBERO"
].is_unique, (
    "Existen códigos de programa Ibero duplicados."
)


print(
    "Programas históricos:",
    len(programas_ibero_historico)
)

print(
    "Programas incorporados desde 2026:",
    len(programas_nuevos_2026)
)

print(
    "Catálogo Ibero actualizado:",
    len(programas_ibero)
)

programas_ibero

Programas históricos: 62
Programas incorporados desde 2026: 15
Catálogo Ibero actualizado: 77


,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,ESTADO_CODIGO
0,2053,LICENCIATURA EN EDUCACION ESPECIAL,DEFINIDO
1,2054,FISIOTERAPIA,DEFINIDO
2,2055,FONOAUDIOLOGIA,DEFINIDO
3,2056,ESPECIALIZACION EN AUDIOLOGIA,DEFINIDO
4,6694,ESPECIALIZACION EN FISIOTERAPIA EN CUIDADO CRI...,DEFINIDO
...,...,...,...
72,PD_MAESTRIA_EN_GERENCIA_DE_PROYECTOS,MAESTRÍA EN GERENCIA DE PROYECTOS,POR DEFINIR
73,119272,DOCTORADO EN EDUCACIÓN,DEFINIDO
74,69836,TÉCNICO LABORAL EN AUXILIAR EN ENFERMERÍA,DEFINIDO
75,PD_MAESTRIA_EN_INGENIERIA_DE_SOFTWARE,MAESTRÍA EN INGENIERÍA DE SOFTWARE,POR DEFINIR


In [43]:
# 5.7 PREPARACIÓN PARA HOMOLOGACIÓN

programas_ibero_match = programas_ibero.copy()

programas_ibero_match["PROGRAMA_IBERO_KEY"] = crear_clave_programa(
    programas_ibero_match["PROGRAMA IBERO"]
)


programas_mercado = dim_programas.copy()

programas_mercado["PROGRAMA_KEY"] = crear_clave_programa(
    programas_mercado["PROGRAMA ACADÉMICO"]
)

#Drop codigos ibero

codigos_ibero_definidos = set(
    programas_ibero.loc[
        ~programas_ibero[
            "CÓDIGO PROGRAMA IBERO"
        ].str.startswith("PD_", na=False),
        "CÓDIGO PROGRAMA IBERO"
    ]
)


programas_mercado = (
    programas_mercado[
        ~programas_mercado[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].isin(codigos_ibero_definidos)
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Programas del mercado a comparar:",
    len(programas_mercado)
)

print(
    "Programas del catálogo Ibero:",
    len(programas_ibero_match)
)

Programas del mercado a comparar: 15500
Programas del catálogo Ibero: 77


In [45]:
# 5.8 VALIDACIÓN DE NOMBRES IBERO REPETIDOS

#validación varios códigos a un mismo programa 
conteo_codigos_por_nombre_ibero = (
    programas_ibero_match
    .groupby("PROGRAMA_IBERO_KEY")[
        "CÓDIGO PROGRAMA IBERO"
    ]
    .nunique()
)


claves_ibero_unicas = set(
    conteo_codigos_por_nombre_ibero[
        conteo_codigos_por_nombre_ibero == 1
    ].index
)


claves_ibero_ambiguas = set(
    conteo_codigos_por_nombre_ibero[
        conteo_codigos_por_nombre_ibero > 1
    ].index
)


print(
    "Nombres Ibero con un solo código:",
    len(claves_ibero_unicas)
)

print(
    "Nombres Ibero asociados a varios códigos:",
    len(claves_ibero_ambiguas)
)

Nombres Ibero con un solo código: 58
Nombres Ibero asociados a varios códigos: 8


In [46]:
# DETALLE DE NOMBRES IBERO CON VARIOS CÓDIGOS

df_nombres_ibero_ambiguos = (
    programas_ibero_match[
        programas_ibero_match[
            "PROGRAMA_IBERO_KEY"
        ].isin(claves_ibero_ambiguas)
    ]
    [
        [
            "CÓDIGO PROGRAMA IBERO",
            "PROGRAMA IBERO",
            "PROGRAMA_IBERO_KEY",
        ]
    ]
    .sort_values(
        [
            "PROGRAMA IBERO",
            "CÓDIGO PROGRAMA IBERO",
        ]
    )
    .reset_index(drop=True)
)

df_nombres_ibero_ambiguos

,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,PROGRAMA_IBERO_KEY
0,110845,ADMINISTRACION FINANCIERA,ADMINISTRACION FINANCIERA
1,108877,ADMINISTRACIÓN FINANCIERA,ADMINISTRACION FINANCIERA
2,102818,CONTADURÍA PÚBLICA,CONTADURIA PUBLICA
3,105216,CONTADURÍA PÚBLICA,CONTADURIA PUBLICA
4,110844,DERECHO,DERECHO
5,115979,DERECHO,DERECHO
6,995,DERECHO,DERECHO
7,103123,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,ESPECIALIZACION EN DESARROLLO INTEGRAL DE LA I...
8,105549,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,ESPECIALIZACION EN DESARROLLO INTEGRAL DE LA I...
9,2053,LICENCIATURA EN EDUCACION ESPECIAL,LICENCIATURA EN EDUCACION ESPECIAL


### 5.9 Coincidencias exactas contra el catálogo Ibero

Primero se identifican coincidencias exactas entre el nombre normalizado del programa SNIES y el catálogo Ibero.

Cuando un nombre Ibero está asociado a un único código, la homologación puede asignarse automáticamente como `EXACTA`.

Cuando el mismo nombre Ibero está asociado a varios códigos, el programa se conserva como `VALIDAR_AMBIGUO` para revisión manual posterior.

In [48]:
# 5.9 COINCIDENCIAS EXACTAS

# Mapa únicamente de nombres Ibero que tienen un solo código
mapa_ibero_exacto = (
    programas_ibero_match[
        programas_ibero_match[
            "PROGRAMA_IBERO_KEY"
        ].isin(claves_ibero_unicas)
    ]
    [
        [
            "CÓDIGO PROGRAMA IBERO",
            "PROGRAMA IBERO",
            "PROGRAMA_IBERO_KEY",
        ]
    ]
    .drop_duplicates(
        subset="PROGRAMA_IBERO_KEY"
    )
)


dim_homologacion = (
    programas_mercado
    .merge(
        mapa_ibero_exacto,
        how="left",
        left_on="PROGRAMA_KEY",
        right_on="PROGRAMA_IBERO_KEY",
        validate="m:1"
    )
)

# Estado inicial
dim_homologacion["ESTADO_HOMOLOGACIÓN"] = "VALIDAR"

# Coincidencias exactas y no ambiguas
mask_exacta = (
    dim_homologacion[
        "CÓDIGO PROGRAMA IBERO"
    ].notna()
)

dim_homologacion.loc[
    mask_exacta,
    "ESTADO_HOMOLOGACIÓN"
] = "EXACTA"

# Coincidencias exactas cuyo nombre Ibero tiene varios códigos
mask_ambigua = (
    dim_homologacion[
        "PROGRAMA_KEY"
    ].isin(claves_ibero_ambiguas)
)

dim_homologacion.loc[
    mask_ambigua,
    "ESTADO_HOMOLOGACIÓN"
] = "VALIDAR_AMBIGUO"


mapa_nombre_ambiguo = (
    programas_ibero_match[
        programas_ibero_match[
            "PROGRAMA_IBERO_KEY"
        ].isin(claves_ibero_ambiguas)
    ]
    .drop_duplicates(
        subset="PROGRAMA_IBERO_KEY"
    )
    .set_index(
        "PROGRAMA_IBERO_KEY"
    )["PROGRAMA IBERO"]
)

dim_homologacion.loc[
    mask_ambigua,
    "PROGRAMA IBERO"
] = (
    dim_homologacion.loc[
        mask_ambigua,
        "PROGRAMA_KEY"
    ]
    .map(mapa_nombre_ambiguo)
)

In [50]:
# 5.10 RESUMEN DE COINCIDENCIAS EXACTAS

resumen_homologacion = (
    dim_homologacion[
        "ESTADO_HOMOLOGACIÓN"
    ]
    .value_counts()
)

resumen_homologacion

ESTADO_HOMOLOGACIÓN
VALIDAR            13041
EXACTA              1680
VALIDAR_AMBIGUO      779
Name: count, dtype: int64

In [52]:
dim_homologacion[
    dim_homologacion[
        "ESTADO_HOMOLOGACIÓN"
    ] == "VALIDAR_AMBIGUO"
][
    [
        "CÓDIGO SNIES DEL PROGRAMA",
        "PROGRAMA ACADÉMICO",
        "PROGRAMA IBERO",
        "ESTADO_HOMOLOGACIÓN",
    ]
].head(10)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,PROGRAMA IBERO,ESTADO_HOMOLOGACIÓN
13,14,PSICOLOGIA,PSICOLOGIA,VALIDAR_AMBIGUO
16,17,DERECHO,DERECHO,VALIDAR_AMBIGUO
160,233,DERECHO,DERECHO,VALIDAR_AMBIGUO
162,235,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,VALIDAR_AMBIGUO
198,294,DERECHO,DERECHO,VALIDAR_AMBIGUO
221,340,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,VALIDAR_AMBIGUO
229,361,DERECHO,DERECHO,VALIDAR_AMBIGUO
232,364,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,VALIDAR_AMBIGUO
266,412,LICENCIATURA EN EDUCACION ESPECIAL,LICENCIATURA EN EDUCACION ESPECIAL,VALIDAR_AMBIGUO
275,434,PSICOLOGIA,PSICOLOGIA,VALIDAR_AMBIGUO


In [53]:
dim_homologacion[
    dim_homologacion[
        "ESTADO_HOMOLOGACIÓN"
    ] == "EXACTA"
][
    [
        "CÓDIGO SNIES DEL PROGRAMA",
        "PROGRAMA ACADÉMICO",
        "CÓDIGO PROGRAMA IBERO",
        "PROGRAMA IBERO",
        "ESTADO_HOMOLOGACIÓN",
    ]
].head(30)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,ESTADO_HOMOLOGACIÓN
7,8,FISIOTERAPIA,2054,FISIOTERAPIA,EXACTA
14,15,TRABAJO SOCIAL,118175,TRABAJO SOCIAL,EXACTA
17,18,ECONOMIA,108466,ECONOMIA,EXACTA
18,19,ADMINISTRACION DE EMPRESAS,110413,ADMINISTRACION DE EMPRESAS,EXACTA
59,70,ESPECIALIZACION EN DERECHO ADMINISTRATIVO,119276,ESPECIALIZACIÓN EN DERECHO ADMINISTRATIVO,EXACTA
89,114,ECONOMIA,108466,ECONOMIA,EXACTA
96,121,INGENIERIA INDUSTRIAL,107162,INGENIERÍA INDUSTRIAL,EXACTA
121,157,LICENCIATURA EN MATEMATICAS,110461,LICENCIATURA EN MATEMATICAS,EXACTA
133,181,LICENCIATURA EN CIENCIAS SOCIALES,110774,LICENCIATURA EN CIENCIAS SOCIALES,EXACTA
135,189,ECONOMIA,108466,ECONOMIA,EXACTA


In [56]:
# CÓDIGOS IBERO POSIBLES PARA NOMBRES AMBIGUOS

mapa_codigos_ambiguos = (
    programas_ibero_match[
        programas_ibero_match[
            "PROGRAMA_IBERO_KEY"
        ].isin(claves_ibero_ambiguas)
    ]
    .groupby("PROGRAMA_IBERO_KEY")[
        "CÓDIGO PROGRAMA IBERO"
    ]
    .agg(lambda x: " | ".join(sorted(set(x.astype(str)))))
)

dim_homologacion[
    "CÓDIGOS_IBERO_CANDIDATOS"
] = (
    dim_homologacion[
        "PROGRAMA_KEY"
    ]
    .map(mapa_codigos_ambiguos)
)

In [55]:
dim_homologacion[
    dim_homologacion[
        "ESTADO_HOMOLOGACIÓN"
    ] == "VALIDAR_AMBIGUO"
][
    [
        "CÓDIGO SNIES DEL PROGRAMA",
        "PROGRAMA ACADÉMICO",
        "PROGRAMA IBERO",
        "CÓDIGOS_IBERO_CANDIDATOS",
        "ESTADO_HOMOLOGACIÓN",
    ]
].head(30)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,PROGRAMA IBERO,CÓDIGOS_IBERO_CANDIDATOS,ESTADO_HOMOLOGACIÓN
13,14,PSICOLOGIA,PSICOLOGIA,105719 | 118307 | 13568,VALIDAR_AMBIGUO
16,17,DERECHO,DERECHO,110844 | 115979 | 995,VALIDAR_AMBIGUO
160,233,DERECHO,DERECHO,110844 | 115979 | 995,VALIDAR_AMBIGUO
162,235,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,102818 | 105216,VALIDAR_AMBIGUO
198,294,DERECHO,DERECHO,110844 | 115979 | 995,VALIDAR_AMBIGUO
221,340,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,102818 | 105216,VALIDAR_AMBIGUO
229,361,DERECHO,DERECHO,110844 | 115979 | 995,VALIDAR_AMBIGUO
232,364,CONTADURIA PUBLICA,CONTADURÍA PÚBLICA,102818 | 105216,VALIDAR_AMBIGUO
266,412,LICENCIATURA EN EDUCACION ESPECIAL,LICENCIATURA EN EDUCACION ESPECIAL,106908 | 2053,VALIDAR_AMBIGUO
275,434,PSICOLOGIA,PSICOLOGIA,105719 | 118307 | 13568,VALIDAR_AMBIGUO


### 5.11 Matching por similitud

Los programas que no presentan una coincidencia exacta se comparan con el catálogo Ibero utilizando similitud textual.

Para cada programa se obtienen los dos candidatos Ibero más similares y sus respectivos scores.

El `SCORE_SIMILITUD` toma valores entre 0 y 100 y mide similitud textual; no representa una probabilidad estadística de equivalencia académica.

La diferencia entre el primer y segundo candidato permite identificar qué tan clara es la sugerencia antes de realizar la validación manual.

In [58]:
# 5.11 MATCHING POR SIMILITUD

from rapidfuzz import process, fuzz


# Catálogo único de nombres Ibero para búsqueda
catalogo_ibero_busqueda = (
    programas_ibero_match[
        [
            "PROGRAMA_IBERO_KEY",
            "PROGRAMA IBERO",
        ]
    ]
    .drop_duplicates(
        subset="PROGRAMA_IBERO_KEY"
    )
    .reset_index(drop=True)
)


claves_ibero = (
    catalogo_ibero_busqueda[
        "PROGRAMA_IBERO_KEY"
    ]
    .tolist()
)

In [59]:
def buscar_candidatos_ibero(programa_key):
    """
    Retorna los dos nombres Ibero más similares
    y sus scores de similitud textual.
    """

    if pd.isna(programa_key) or not programa_key:
        return pd.Series(
            [pd.NA, pd.NA, pd.NA, pd.NA]
        )

    resultados = process.extract(
        programa_key,
        claves_ibero,
        scorer=fuzz.WRatio,
        limit=2
    )

    if not resultados:
        return pd.Series(
            [pd.NA, pd.NA, pd.NA, pd.NA]
        )

    key_1, score_1, _ = resultados[0]

    if len(resultados) > 1:
        key_2, score_2, _ = resultados[1]
    else:
        key_2, score_2 = pd.NA, pd.NA

    return pd.Series(
        [
            key_1,
            round(score_1, 2),
            key_2,
            round(score_2, 2)
            if pd.notna(score_2)
            else pd.NA,
        ]
    )

In [60]:
#Solo los que están para validar

mask_validar = (
    dim_homologacion[
        "ESTADO_HOMOLOGACIÓN"
    ] == "VALIDAR"
)


resultados_fuzzy = (
    dim_homologacion.loc[
        mask_validar,
        "PROGRAMA_KEY"
    ]
    .apply(buscar_candidatos_ibero)
)


resultados_fuzzy.columns = [
    "CANDIDATO_1_KEY",
    "SCORE_CANDIDATO_1",
    "CANDIDATO_2_KEY",
    "SCORE_CANDIDATO_2",
]


dim_homologacion.loc[
    mask_validar,
    resultados_fuzzy.columns
] = resultados_fuzzy.values

In [61]:
#Nombres reales 

mapa_nombre_ibero = (
    catalogo_ibero_busqueda
    .set_index(
        "PROGRAMA_IBERO_KEY"
    )["PROGRAMA IBERO"]
)


dim_homologacion["CANDIDATO_1_IBERO"] = (
    dim_homologacion[
        "CANDIDATO_1_KEY"
    ]
    .map(mapa_nombre_ibero)
)


dim_homologacion["CANDIDATO_2_IBERO"] = (
    dim_homologacion[
        "CANDIDATO_2_KEY"
    ]
    .map(mapa_nombre_ibero)
)

In [63]:
# DIFERENCIA ENTRE LOS DOS MEJORES CANDIDATOS

dim_homologacion["DIFERENCIA_SCORE"] = (
    pd.to_numeric(
        dim_homologacion[
            "SCORE_CANDIDATO_1"
        ],
        errors="coerce"
    )
    -
    pd.to_numeric(
        dim_homologacion[
            "SCORE_CANDIDATO_2"
        ],
        errors="coerce"
    )
)

df_revision_similitud = (
    dim_homologacion[
        dim_homologacion[
            "ESTADO_HOMOLOGACIÓN"
        ] == "VALIDAR"
    ]
    [
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
            "CANDIDATO_1_IBERO",
            "SCORE_CANDIDATO_1",
            "CANDIDATO_2_IBERO",
            "SCORE_CANDIDATO_2",
            "DIFERENCIA_SCORE",
        ]
    ]
    .sort_values(
        [
            "SCORE_CANDIDATO_1",
            "DIFERENCIA_SCORE",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


df_revision_similitud.head(30)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,CANDIDATO_1_IBERO,SCORE_CANDIDATO_1,CANDIDATO_2_IBERO,SCORE_CANDIDATO_2,DIFERENCIA_SCORE
0,52565,ESPECIALIZACIÓN EN GERENCIA DE TALENTO HUMANO,ESPECIALIZACIÓN EN GERENCIA DEL TALENTO HUMANO,98.90,ESPECIALIZACION EN ALTA GERENCIA,86.95,11.95
1,111250,ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIO,ESPECIALIZACION EN INTELIGENCIA DE NEGOCIOS,98.82,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,85.50,13.32
2,105325,ESPECIALIZACIÓN EN GERENCIA DE PROYECTO,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,98.73,ESPECIALIZACION EN ALTA GERENCIA,86.95,11.78
3,117799,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,97.84,LICENCIATURA EN EDUCACION ESPECIAL,85.50,12.34
4,107406,INGENIIERIA DE SOFTWARE,INGENIERÍA DE SOFTWARE,97.78,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,85.50,12.28
5,107398,ESPECIALIZACIÓN EN GERENCIA DE SEGURIDAD Y SAL...,ESPECIALIZACIÓN EN GERENCIA DE LA SEGURIDAD Y ...,97.64,SEGURIDAD Y SALUD EN EL TRABAJO,90.00,7.64
6,110249,ESPECIALIZACIÓN EN GERENCIA DE SEGURIDAD Y SAL...,ESPECIALIZACIÓN EN GERENCIA DE LA SEGURIDAD Y ...,97.64,SEGURIDAD Y SALUD EN EL TRABAJO,90.00,7.64
7,110807,ESPECIALIZACIÓN EN GERENCIA DE SEGURIDAD Y SAL...,ESPECIALIZACIÓN EN GERENCIA DE LA SEGURIDAD Y ...,97.64,SEGURIDAD Y SALUD EN EL TRABAJO,90.00,7.64
8,105901,LICENCIATURA EN LENGUA EXTRANJERA INGLES,LICENCIATURA EN LENGUAS EXTRANJERAS INGLÉS,97.56,ESPECIALIZACIÓN EN DESARROLLO INTEGRAL DE LA I...,85.50,12.06
9,111418,ESPECIALIZACIÓN EN DERECHO ADMINSITRATIVO,ESPECIALIZACIÓN EN DERECHO ADMINISTRATIVO,97.56,DERECHO,90.00,7.56


### 5.12 Preparación de la base para revisión manual

Para facilitar la validación de las homologaciones propuestas, se incorporan atributos académicos observados en el histórico SNIES:

- `NIVEL DE FORMACIÓN`
- `MODALIDAD`
- `DESC CINE CAMPO DETALLADO`

Como un mismo código SNIES puede aparecer en múltiples años y registros, no se realiza un cruce directo contra el histórico. Primero se consolida una única fila por código, conservando los distintos valores observados.

La base de revisión contiene los programas pendientes de validación, los candidatos Ibero sugeridos, los scores de similitud y campos para registrar la decisión manual.

In [64]:
# 5.12 CONTEXTO HISTÓRICO PARA REVISIÓN MANUAL

def unir_valores_unicos(serie):
    """
    Consolida los valores distintos observados en el histórico
    sin seleccionar arbitrariamente uno.
    """

    valores = (
        serie
        .dropna()
        .astype("string")
        .str.strip()
    )

    valores = valores[
        valores.ne("")
    ]

    return " | ".join(
        sorted(valores.unique())
    )


contexto_programas = (
    df_historico
    .groupby(
        "CÓDIGO SNIES DEL PROGRAMA",
        as_index=False
    )
    .agg(
        {
            "NIVEL DE FORMACIÓN":
                unir_valores_unicos,

            "MODALIDAD":
                unir_valores_unicos,

            "DESC CINE CAMPO DETALLADO":
                unir_valores_unicos,
        }
    )
)

contexto_programas.head()

,CÓDIGO SNIES DEL PROGRAMA,NIVEL DE FORMACIÓN,MODALIDAD,DESC CINE CAMPO DETALLADO
0,1,Universitaria | Universitario,Presencial,Producción agrícola y ganadera
1,10,Universitaria | Universitario,Presencial,"Fisioterapia, Fonoaudiología, Terapia ocupacio..."
2,100,Maestría,Presencial,Construcción e ingeniería civil
3,1001,Especialización universitaria,Presencial,Derecho
4,10010,Especialización médico quirúrgica,Presencial,Medicina


In [73]:
# 5.12 BASE MAESTRA DE HOMOLOGACIÓN

df_homologacion_final = (
    dim_homologacion
    .merge(
        contexto_programas,
        how="left",
        on="CÓDIGO SNIES DEL PROGRAMA",
        validate="1:1"
    )
    .copy()
)

# CAMPOS DE RESULTADO Y VALIDACIÓN MANUAL

df_homologacion_final[
    "DECISIÓN_MANUAL"
] = pd.NA

df_homologacion_final[
    "CÓDIGO_IBERO_VALIDADO"
] = pd.NA

df_homologacion_final[
    "PROGRAMA_IBERO_VALIDADO"
] = pd.NA

df_homologacion_final[
    "OBSERVACIÓN"
] = pd.NA

# Las coincidencias exactas ya quedan resueltas automáticamente

mask_exacta_final = (
    df_homologacion_final[
        "ESTADO_HOMOLOGACIÓN"
    ] == "EXACTA"
)


df_homologacion_final.loc[
    mask_exacta_final,
    "DECISIÓN_MANUAL"
] = "NO REQUIERE"


df_homologacion_final.loc[
    mask_exacta_final,
    "CÓDIGO_IBERO_VALIDADO"
] = (
    df_homologacion_final.loc[
        mask_exacta_final,
        "CÓDIGO PROGRAMA IBERO"
    ]
)


df_homologacion_final.loc[
    mask_exacta_final,
    "PROGRAMA_IBERO_VALIDADO"
] = (
    df_homologacion_final.loc[
        mask_exacta_final,
        "PROGRAMA IBERO"
    ]
)

In [76]:
# ORDEN DE COLUMNAS DE LA BASE MAESTRA

columnas_homologacion = [
    "CÓDIGO SNIES DEL PROGRAMA",
    "PROGRAMA ACADÉMICO",

    "NIVEL DE FORMACIÓN",
    "MODALIDAD",
    "DESC CINE CAMPO DETALLADO",

    "CÓDIGO PROGRAMA IBERO",
    "PROGRAMA IBERO",

    "CÓDIGOS_IBERO_CANDIDATOS",

    "CANDIDATO_1_IBERO",
    "SCORE_CANDIDATO_1",

    "CANDIDATO_2_IBERO",
    "SCORE_CANDIDATO_2",

    "DIFERENCIA_SCORE",

    "ESTADO_HOMOLOGACIÓN",

    "DECISIÓN_MANUAL",
    "CÓDIGO_IBERO_VALIDADO",
    "PROGRAMA_IBERO_VALIDADO",
    "OBSERVACIÓN",
]


columnas_homologacion = [
    columna
    for columna in columnas_homologacion
    if columna in df_homologacion_final.columns
]


df_homologacion_final = (
    df_homologacion_final[
        columnas_homologacion
    ]
    .copy()
)
df_homologacion_final = (
    df_homologacion_final
    .sort_values(
        [
            "ESTADO_HOMOLOGACIÓN",
            "SCORE_CANDIDATO_1",
            "DIFERENCIA_SCORE",
        ],
        ascending=[
            True,
            False,
            False,
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


df_homologacion_final.head(10)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,NIVEL DE FORMACIÓN,MODALIDAD,DESC CINE CAMPO DETALLADO,CÓDIGO PROGRAMA IBERO,PROGRAMA IBERO,CÓDIGOS_IBERO_CANDIDATOS,CANDIDATO_1_IBERO,SCORE_CANDIDATO_1,CANDIDATO_2_IBERO,SCORE_CANDIDATO_2,DIFERENCIA_SCORE,ESTADO_HOMOLOGACIÓN,DECISIÓN_MANUAL,CÓDIGO_IBERO_VALIDADO,PROGRAMA_IBERO_VALIDADO,OBSERVACIÓN
0,8,FISIOTERAPIA,Universitaria | Universitario,Presencial,"Fisioterapia, Fonoaudiología, Terapia ocupacio...",2054,FISIOTERAPIA,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,2054,FISIOTERAPIA,<NA>
1,15,TRABAJO SOCIAL,Universitaria | Universitario,Presencial,Trabajo social,118175,TRABAJO SOCIAL,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,118175,TRABAJO SOCIAL,<NA>
2,18,ECONOMIA,Universitaria | Universitario,Presencial,Economía,108466,ECONOMIA,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,108466,ECONOMIA,<NA>
3,19,ADMINISTRACION DE EMPRESAS,Universitaria | Universitario,Presencial,Gestión y administración,110413,ADMINISTRACION DE EMPRESAS,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,110413,ADMINISTRACION DE EMPRESAS,<NA>
4,70,ESPECIALIZACION EN DERECHO ADMINISTRATIVO,Especialización universitaria,Presencial,Derecho,119276,ESPECIALIZACIÓN EN DERECHO ADMINISTRATIVO,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,119276,ESPECIALIZACIÓN EN DERECHO ADMINISTRATIVO,<NA>
5,114,ECONOMIA,Universitaria | Universitario,Presencial,Economía,108466,ECONOMIA,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,108466,ECONOMIA,<NA>
6,121,INGENIERIA INDUSTRIAL,Universitaria | Universitario,Presencial,Ingeniería y profesiones afines no clasificada...,107162,INGENIERÍA INDUSTRIAL,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,107162,INGENIERÍA INDUSTRIAL,<NA>
7,157,LICENCIATURA EN MATEMATICAS,Universitaria | Universitario,Presencial,Ciencias de la educación,110461,LICENCIATURA EN MATEMATICAS,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,110461,LICENCIATURA EN MATEMATICAS,<NA>
8,181,LICENCIATURA EN CIENCIAS SOCIALES,Universitaria | Universitario,Presencial,Ciencias de la educación,110774,LICENCIATURA EN CIENCIAS SOCIALES,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,110774,LICENCIATURA EN CIENCIAS SOCIALES,<NA>
9,189,ECONOMIA,Universitaria | Universitario,Presencial,Economía,108466,ECONOMIA,<NA>,<NA>,NaN,<NA>,NaN,NaN,EXACTA,NO REQUIERE,108466,ECONOMIA,<NA>


In [77]:
# 5.13 EXPORTACIÓN DE LA BASE MAESTRA DE HOMOLOGACIÓN

OUTPUT_HOMOLOGACION = (
    PROCESSED
    / "03_Homologacion_Programas_Ibero.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_HOMOLOGACION,
    engine="openpyxl"
) as writer:

    # Base completa de homologación
    df_homologacion_final.to_excel(
        writer,
        sheet_name="Homologacion",
        index=False
    )

    # Catálogo Ibero de referencia
    programas_ibero[
        [
            "CÓDIGO PROGRAMA IBERO",
            "PROGRAMA IBERO",
            "ESTADO_CODIGO",
        ]
    ].to_excel(
        writer,
        sheet_name="Catalogo_Ibero",
        index=False
    )


print("Archivo generado correctamente:")
print(OUTPUT_HOMOLOGACION)

Archivo generado correctamente:
C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed\03_Homologacion_Programas_Ibero.xlsx
